In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "Password123"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({"animal_type": {"$exists": True}}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
print(len(df.to_dict(orient='records')))
print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Div(
        style={
            "height": "150px", #ensures image fit
            "display": "flex",
            "alignItems": "center",
            "justifyContent": "center",
            "position": "relative"
        },
        children=[

            html.A(
                html.Img(
                    src='data:image/png;base64,{}'.format(encoded_image.decode()),
                    style={
                        'height': '150px'
                    }
                ),
                href='https://www.snhu.edu',
                target='_blank'
            ),
            html.Div(
                style={
                    "display": "flex",
                    "flexDirection": "column",
                    "alignItems": "center"
                },
                children=[
                    html.H1(
                        "Grazioso Salvare Dashboard",
                        style={"margin": "0"}
                    ),
                    html.H2(
                        "CS-340 Final",
                        style={"margin": "0"}
                    ),
                    html.H3(
                        "Mathew Masar",
                        style={"margin": "0"}
                    ),
                ]
            )
        ]
    ),
    html.Hr(),
    html.Div(
        
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
        style={
            "display": "flex",
            "justifyContent": "space-between",
            "alignItems": "center",
            "padding": "10px 20px"
        },
        children=[
            # LEFT: Radio buttons
            dcc.RadioItems(
                id="filter-type",
                options=[
                    {"label": "Water Rescue", "value": "water"},
                    {"label": "Mountain / Wilderness", "value": "mountain"},
                    {"label": "Disaster / Tracking", "value": "disaster"},
                    {"label": "Default / Reset", "value": "reset"},
                ],
                value="reset",
                inline=True,
                labelStyle={
                "marginRight": "30px"
                }
            ),

            # RIGHT: Total results text
            html.Div(
                id="result-count",
                style={"fontWeight": "bold"}
            )
        ]
        
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 

        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        page_action="native",
        page_current=0,
        page_size=10,
        row_selectable="single",
        selected_rows=[0],   
                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################


@app.callback(
    [Output("datatable-id", "data"),
     Output("result-count", "children")],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):

    # Default/Reset == match everything
    if filter_type == "reset":
        query = {"animal_type": {"$exists": True}}

    elif filter_type == "water":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "mountain":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                              "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "disaster":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                              "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {"animal_type": {"$exists": True}}

    results = db.read(query)
    dff = pd.DataFrame.from_records(results)

    # Drop _id safely
    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    data = dff.to_dict("records")
    count_text = f"Total Results: {len(data)}"

    return data, count_text
    


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [
        Input('datatable-id', "derived_virtual_data"),
        Input("filter-type", "value")
    ]
)
def update_graphs(viewData, filter_type):
    ###FIX ME ####
    if viewData is None or len(viewData) == 0:
        return html.Div("No available data to display.")
    
    dff = pd.DataFrame.from_dict(viewData)
    
    
    # --- IF RESET: show category breakdown chart ---
    if filter_type == "reset":
        water_query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

        mountain_query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                              "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

        disaster_query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                              "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

        water_count = len(db.read(water_query))
        mountain_count = len(db.read(mountain_query))
        disaster_count = len(db.read(disaster_query))

        categorized = water_count + mountain_count + disaster_count

        summary = pd.DataFrame({
            "Category": ["Water Rescue", "Mountain/Wilderness", "Disaster/Tracking"],
            "Count": [water_count, mountain_count, disaster_count]
        })

        fig = px.pie(
            summary,
            names="Category",
            values="Count",
        )
        
        # Center title
        fig.update_layout(
            title={
                "text": f"Rescue Distribution Amongst Programs<br><sup>Total Eligible Rescues: {categorized}</sup>",
                "x": 0.5,
                "xanchor": "center"
            }
        )
        
        return [dcc.Graph(figure=fig)]
    
    
    # --- ELSE: Show individual category breakdown chart ---
    program_map = {
        "water": "Water Rescue",
        "mountain": "Mountain / Wilderness",
        "disaster": "Disaster / Tracking"
    }
    
    program_type = program_map.get(filter_type)
    
    fig = px.pie(
        dff,
        names="breed",
        title=f"Breed Percentages of {program_type} Program"
    )
    
    #Center title
    fig.update_layout(
    title_x=0.5
    )

    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Used to update center of map for each selection
    selected_lat = dff.iloc[row, 13]
    selected_lon = dff.iloc[row, 14]
    
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[selected_lat, selected_lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.B("Animal Record"),
                    html.Br(),
                    html.Span(f"Animal ID: {dff.iloc[row].get('animal_id')}"),
                    html.Br(),
                    html.Span(f"Name: {dff.iloc[row].get('name') or 'N/A'}"),
                    html.Br(),
                    html.Span(f"Type: {dff.iloc[row].get('animal_type')}"),
                    html.Br(),
                    html.Span(f"Breed: {dff.iloc[row].get('breed')}"),
                    html.Br(),
                    html.Span(f"Outcome: {dff.iloc[row].get('outcome_type')}"),
                    html.Br(),
                    html.Span(f"Sex: {dff.iloc[row].get('sex_upon_outcome')}"),
                    html.Br(),
                    html.Span(f"Age: {dff.iloc[row].get('age_upon_outcome')}"),
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

10009
Index(['rec_num', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed',
       'color', 'date_of_birth', 'datetime', 'monthyear', 'name',
       'outcome_subtype', 'outcome_type', 'sex_upon_outcome', 'location_lat',
       'location_long', 'age_upon_outcome_in_weeks'],
      dtype='object')
Dash app running on https://forgetglass-dietnothing-3000.codio.io/proxy/8050/
